# 01 — Data Preprocessing

**Dataset:** Online Retail II (UCI / Kaggle)
Covers two years of transactions (Dec 2009 – Dec 2011) for a UK-based online retailer.

**Goals:**
- Load and inspect both yearly sheets
- Remove invalid / cancelled records
- Engineer time and price features
- Save clean CSVs for downstream notebooks


In [1]:
import os, warnings
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('outputs/figures', exist_ok=True)
print(f'Working dir: {os.getcwd()}')
import pandas as pd
import numpy as np

Working dir: /Users/neelp03/Desktop/school/cmpe255/CMPE255-project


## 1. Load Raw Data

In [2]:
print("Loading Excel file — this may take ~60 seconds...")
df1 = pd.read_excel('data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('data/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([df1, df2], ignore_index=True)
df.columns = df.columns.str.strip()
df = df.rename(columns={'Customer ID': 'CustomerID'})
print(f"Raw shape: {df.shape}")
df.head()

Loading Excel file — this may take ~60 seconds...


Raw shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print("Column types:")
print(df.dtypes)
print(f"\nDate range: {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"\nMissing values:")
print(df.isnull().sum())

Column types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
CustomerID            float64
Country                   str
dtype: object

Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00

Missing values:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
CustomerID     243007
Country             0
dtype: int64


## 2. Data Cleaning

In [4]:
original = len(df)

# Remove cancelled invoices (Invoice starts with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove rows with non-positive Quantity or Price
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Remove rows with missing Description
df = df.dropna(subset=['Description'])
df['Description'] = df['Description'].str.strip().str.upper()

removed = original - len(df)
print(f"Removed {removed:,} invalid rows ({removed/original*100:.1f}% of raw data)")
print(f"Clean shape: {df.shape}")
print(f"\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Removed 25,701 invalid rows (2.4% of raw data)
Clean shape: (1041670, 8)

Remaining nulls:
CustomerID    236121
dtype: int64


## 3. Feature Engineering

In [5]:
df['TotalPrice']  = df['Quantity'] * df['Price']
df['Year']        = df['InvoiceDate'].dt.year
df['Month']       = df['InvoiceDate'].dt.month
df['DayOfWeek']   = df['InvoiceDate'].dt.dayofweek   # 0=Mon, 6=Sun
df['Hour']        = df['InvoiceDate'].dt.hour

print("New features added: TotalPrice, Year, Month, DayOfWeek, Hour")
df[['Invoice','InvoiceDate','TotalPrice','Year','Month','DayOfWeek','Hour']].head()

New features added: TotalPrice, Year, Month, DayOfWeek, Hour


,Invoice,InvoiceDate,TotalPrice,Year,Month,DayOfWeek,Hour
0,489434,2009-12-01 07:45:00,83.4,2009,12,1,7
1,489434,2009-12-01 07:45:00,81.0,2009,12,1,7
2,489434,2009-12-01 07:45:00,81.0,2009,12,1,7
3,489434,2009-12-01 07:45:00,100.8,2009,12,1,7
4,489434,2009-12-01 07:45:00,30.0,2009,12,1,7


## 4. Save Cleaned Data

In [6]:
# Full cleaned dataset (includes rows without CustomerID — needed for basket analysis)
df.to_csv('data/cleaned_retail.csv', index=False)
print(f"Saved cleaned_retail.csv  →  {df.shape}")

# Customer-level dataset (only rows with a known CustomerID)
df_cust = df.dropna(subset=['CustomerID']).copy()
df_cust['CustomerID'] = df_cust['CustomerID'].astype(int)
df_cust.to_csv('data/cleaned_retail_customers.csv', index=False)
print(f"Saved cleaned_retail_customers.csv  →  {df_cust.shape}")

print(f"\n{'='*45}")
print(f"  Unique invoices  : {df['Invoice'].nunique():>8,}")
print(f"  Unique products  : {df['StockCode'].nunique():>8,}")
print(f"  Unique customers : {df_cust['CustomerID'].nunique():>8,}")
print(f"  Countries        : {df['Country'].nunique():>8}")
print(f"  Total revenue    : £{df_cust['TotalPrice'].sum():>12,.0f}")
print(f"{'='*45}")
print("Preprocessing complete!")

Saved cleaned_retail.csv  →  (1041670, 13)


Saved cleaned_retail_customers.csv  →  (805549, 13)

  Unique invoices  :   40,077
  Unique products  :    4,917
  Unique customers :    5,878
  Countries        :       43
  Total revenue    : £  17,743,429
Preprocessing complete!
